In [3]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer

import pickle

from sklearn.metrics import root_mean_squared_error

In [5]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("nyc-taxi-experiment")

2026/11/04 15:32:20 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1793802740656, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793802740656, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [10]:
def read_parquet(file_link):
    df = pd.read_parquet(file_link)

    
    
    df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df['duration'] = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df = df[df.duration >= 1]
    df = df[df.duration <= 60]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    return df

In [11]:
df_train = read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_eval = read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [12]:
categorical = ['PU_DO'] #['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

eval_dicts = df_eval[categorical + numerical].to_dict(orient='records')
X_eval = dv.transform(eval_dicts)

In [14]:
import numpy as np

X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

X_eval.indices = X_eval.indices.astype(np.int32)
X_eval.indptr = X_eval.indptr.astype(np.int32)

In [15]:
target = 'duration'
y_train = df_train[target].values
y_eval = df_eval[target].values

In [18]:
import xgboost as xgb

In [23]:
from pathlib import Path

In [24]:
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [25]:
with mlflow.start_run():

    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_eval, label=y_eval)

    best_params = {
        "learning_rate" : 0.2948646839243271,
        "max_depth" : 30,
        "min_child_weight" : 1.9805346649872448,
        "objective" : "reg:squarederror",
        "reg_alpha" : 0.06275960619157453,
        "reg_lambda" : 0.22388496105060088,
        "seed" : 42
    }

    mlflow.log_params(best_params)
    
    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=30,
        evals=[(valid, "validation")],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_eval, y_pred)
    mlflow.log_metric("RMSE", rmse)

    with open("../models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact("../models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:9.92391
[1]	validation-rmse:8.52365
[2]	validation-rmse:7.69333
[3]	validation-rmse:7.21505
[4]	validation-rmse:6.93944
[5]	validation-rmse:6.77711
[6]	validation-rmse:6.68196
[7]	validation-rmse:6.62197
[8]	validation-rmse:6.58173
[9]	validation-rmse:6.55463
[10]	validation-rmse:6.53570
[11]	validation-rmse:6.52075
[12]	validation-rmse:6.51208
[13]	validation-rmse:6.50465
[14]	validation-rmse:6.50212
[15]	validation-rmse:6.49959
[16]	validation-rmse:6.49611
[17]	validation-rmse:6.49361
[18]	validation-rmse:6.48940
[19]	validation-rmse:6.48529
[20]	validation-rmse:6.48214
[21]	validation-rmse:6.47965
[22]	validation-rmse:6.47731
[23]	validation-rmse:6.47499
[24]	validation-rmse:6.47183
[25]	validation-rmse:6.46963
[26]	validation-rmse:6.46684
[27]	validation-rmse:6.46545
[28]	validation-rmse:6.46225
[29]	validation-rmse:6.46082


2026/11/04 15:44:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/04 15:44:09 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run vaunted-crab-553 at: http://127.0.0.1:5000/#/experiments/1/runs/8babc83d09ab446a831c901c48adebf8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [17]:
logged_model = 'models:/m-27d6a2620cf343fa8faf5bc5848cf85b'

loaded_model = mlflow.pyfunc.load_model(logged_model)

In [18]:
xgboost_model = mlflow.xgboost.load_model(logged_model)
xgboost_model

In [19]:
y_pred = xgboost_model.predict(valid)
y_pred[:10]

array([14.586527 ,  7.2014832, 14.387785 , 24.29462  ,  9.860293 ,
       17.205248 , 11.048476 ,  8.38429  ,  9.754515 , 20.591978 ],
      dtype=float32)